<a href="https://colab.research.google.com/github/pmlade23-droid/Analyse-Sentiment-IMDb./blob/main/The_Model_Mirage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 0: Environment Setup
Importing core libraries and tools for data handling, modeling, and evaluation.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Environment ready. Let's build the mirage.")

Environment ready. Let's build the mirage.


### Step 1: Generating Our Controlled Dataset
Creating a synthetic dataset with injected missing values to control our data experiment.

In [3]:
from sklearn.datasets import make_regression

# Generate synthetic data
X_raw, y_raw = make_regression(n_samples=1000, n_features=10, noise=0.5, random_state=42)
df = pd.DataFrame(X_raw, columns=[f"feat_{i}" for i in range(10)])
df['target'] = y_raw

# Inject 10% missing values
np.random.seed(42)
mask = np.random.rand(*df.iloc[:, :-1].shape) < 0.1
df.iloc[:, :-1] = df.iloc[:, :-1].mask(mask)

print(f"Dataset shape: {df.shape}")

Dataset shape: (1000, 11)


### Step 2: The Flawed Approach (Data Leakage)
Imputing missing values globally before splitting. This leaks validation info into training and artificially boosts performance.



In [4]:
# 1. Impute globally (THE MISTAKE)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df.drop(columns=['target']))
y = df['target'].values

# 2. Split afterward
X_train, X_val, y_train, y_val = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# 3. Train and evaluate
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
flawed_score = mean_absolute_error(y_val, model.predict(X_val))

print(f"Flawed Model MAE (With Leakage): {flawed_score:.4f}")

Flawed Model MAE (With Leakage): 51.0112
